# Loan (UCI credit default): the well-behaved one that still lies

*Notebook journey for §2 of the dissertation. Every number the chapter uses is produced here, and each step prints what it did. We follow the deep-learning basics faithfully, then look hard at what the number actually measures.*

## a. Reading the problem

In [1]:
import numpy as np, os

# the loan data (UCI credit-card default): each row is one client,
# 23 numbers about them, and whether they defaulted.
DATA = "data/loan_uci350.csv" if os.path.exists("data/loan_uci350.csv") else "../data/loan_uci350.csv"
a = np.loadtxt(DATA, delimiter=",", skiprows=1)   # the header row is just column indices 0..23
X, y = a[:, :-1], a[:, -1].astype(int)

print("clients :", len(y))
print("features:", X.shape[1], "(per client)")
defaulters = int((y == 1).sum())
print("defaulters:", defaulters, f"({defaulters / len(y):.1%})")

# read ONE client top to bottom, so we see what a single row is
i = 0
c = X[i]
print(f"\none client (row {i}):")
print("  credit limit   :", int(c[0]))
print("  age            :", int(c[4]))
print("  latest bill    :", int(c[11]))
print("  latest payment :", int(c[17]))
print("  ...23 numbers in all...")
print("  defaulted?     :", int(y[i]), "(1 = yes)")

# separate people, no time column, few repeats; and the do-nothing baseline
print("\nexact-duplicate rows:", len(a) - len(np.unique(a, axis=0)), "of", len(a))
print("always-guess 'no default' is right:", round(float((y == 0).mean()), 4))

clients : 30000
features: 23 (per client)
defaulters: 6636 (22.1%)

one client (row 0):
  credit limit   : 20000
  age            : 24
  latest bill    : 3913
  latest payment : 0
  ...23 numbers in all...
  defaulted?     : 1 (1 = yes)

exact-duplicate rows: 35 of 30000
always-guess 'no default' is right: 0.7788


## b. By the book, and does the split matter?

We build the network from Section 1, hold out a fifth for the test, scale on the training side, and train. But first the worry: are these rows really interchangeable? We cut the deck many ways, ten random shuffles and a stratified split, and see whether the number moves. Then one more cut, by raw CSV row order, as a check.

In [2]:
# the model: the same small network from Section 1 (one hidden layer, ReLU, softmax)
def init(d, width, K, rng):
    return [rng.standard_normal((d, width)) * 0.1, np.zeros(width),
            rng.standard_normal((width, K)) * 0.1, np.zeros(K)]

def forward(X, p):
    W1, b1, W2, b2 = p
    a = X @ W1 + b1; h = np.maximum(a, 0.0); z = h @ W2 + b2
    return a, h, z

def softmax(z):
    z = z - z.max(1, keepdims=True); e = np.exp(z); return e / e.sum(1, keepdims=True)

def accuracy(p, X, y):
    return float((forward(X, p)[2].argmax(1) == y).mean())

def train(Xtr, ytr, width, lr, epochs, K, rng):
    p = init(Xtr.shape[1], width, K, rng); n = len(ytr)
    Y = np.zeros((n, K)); Y[np.arange(n), ytr] = 1.0
    for _ in range(epochs):
        a, h, z = forward(Xtr, p)
        dz = (softmax(z) - Y) / n
        W1, b1, W2, b2 = p
        dW2 = h.T @ dz; db2 = dz.sum(0)
        da  = (dz @ W2.T) * (a > 0)
        dW1 = Xtr.T @ da; db1 = da.sum(0)
        p = [W1 - lr*dW1, b1 - lr*db1, W2 - lr*dW2, b2 - lr*db2]
    return p

def standardize(Xtr, Xte):                # fit on the training side only
    m, s = Xtr.mean(0), Xtr.std(0) + 1e-9
    return (Xtr - m) / s, (Xte - m) / s

def score(how, seed, width=16, lr=0.3, epochs=300):
    n = len(y); ntest = n // 5; rng = np.random.default_rng(seed)
    if how == "stratified":               # keep the 22% default rate on each side
        te = np.concatenate([rng.permutation(np.where(y == cc)[0])[:int(round(ntest * np.mean(y == cc)))] for cc in (0, 1)])
        tr = np.setdiff1d(np.arange(n), te)
    elif how == "position":               # last fifth by CSV row order
        te = np.arange(n - ntest, n); tr = np.arange(n - ntest)
    else:                                 # plain random shuffle
        idx = rng.permutation(n); te, tr = idx[:ntest], idx[ntest:]
    Xtr, Xte = standardize(X[tr], X[te])
    p = train(Xtr, y[tr], width, lr, epochs, 2, np.random.default_rng(seed + 1))
    return accuracy(p, Xte, y[te])

shuffles = [score("shuffle", s) for s in range(10)]
print("10 random shuffles :", [round(v, 3) for v in shuffles])
print("              mean  :", round(float(np.mean(shuffles)), 3), " sd", round(float(np.std(shuffles)), 3))
print("stratified split    :", round(score("stratified", 0), 3))
print("position split (CSV order):", round(score("position", 0), 3))

10 random shuffles : [0.819, 0.811, 0.819, 0.817, 0.814, 0.824, 0.813, 0.818, 0.814, 0.821]
              mean  : 0.817  sd 0.004


stratified split    : 0.816


position split (CSV order): 0.83


## c. But did it catch the defaulters?

The 0.817 looks fine. But the whole point was to find the clients who will default. So on one honest split we stop reading the single number and count what the model actually did, group by group.

In [3]:
# one honest random split; stop reading the single number and count what happened to the defaulters
rng = np.random.default_rng(0)
n = len(y); ntest = n // 5
idx = rng.permutation(n); te, tr = idx[:ntest], idx[ntest:]
Xtr, Xte = standardize(X[tr], X[te])
p = train(Xtr, y[tr], 16, 0.3, 300, 2, np.random.default_rng(1))
yhat = forward(Xte, p)[2].argmax(1)
yt = y[te]

TP = int(((yhat == 1) & (yt == 1)).sum())   # defaulters caught
FN = int(((yhat == 0) & (yt == 1)).sum())   # defaulters missed
TN = int(((yhat == 0) & (yt == 0)).sum())   # payers correctly cleared
FP = int(((yhat == 1) & (yt == 0)).sum())   # payers wrongly flagged
acc = (TP + TN) / len(yt)
recall = TP / (TP + FN)                      # of the real defaulters, the fraction caught

print("accuracy on this test set:", round(acc, 3))
print("what it did with each group:")
print(f"  real defaulters: {TP + FN}  ->  caught {TP}, missed {FN}")
print(f"  real payers    : {TN + FP}  ->  cleared {TN}, flagged {FP}")
print("recall (defaulters caught):", round(recall, 3))

accuracy on this test set: 0.819
what it did with each group:
  real defaulters: 1353  ->  caught 459, missed 894
  real payers    : 4647  ->  cleared 4454, flagged 193
recall (defaulters caught): 0.339


## d. The number we keep

Accuracy counts every client the same, so the rare defaulters vanish inside the majority. Balanced accuracy scores the defaulters and the payers on their own and averages the two, so the small class counts as much as the big one. Here is both, over several honest splits.

In [4]:
# balanced accuracy: score each class on its own and average, so the rare class counts equally
def scores(seed):
    rng = np.random.default_rng(seed); n = len(y); ntest = n // 5
    idx = rng.permutation(n); te, tr = idx[:ntest], idx[ntest:]
    Xtr, Xte = standardize(X[tr], X[te])
    p = train(Xtr, y[tr], 16, 0.3, 300, 2, np.random.default_rng(seed + 1))
    yhat = forward(Xte, p)[2].argmax(1); yt = y[te]
    recall = ((yhat == 1) & (yt == 1)).sum() / (yt == 1).sum()   # of defaulters, fraction caught
    spec   = ((yhat == 0) & (yt == 0)).sum() / (yt == 0).sum()   # of payers, fraction cleared
    acc    = (yhat == yt).mean()
    return float(acc), float(recall), float(spec), float((recall + spec) / 2)

rows = [scores(s) for s in range(5)]
acc  = np.mean([r[0] for r in rows]); rec  = np.mean([r[1] for r in rows])
spec = np.mean([r[2] for r in rows]); bal  = np.mean([r[3] for r in rows])
print("over 5 honest splits, mean:")
print("  plain accuracy      :", round(acc, 3))
print("  recall (defaulters) :", round(rec, 3))
print("  specificity (payers):", round(spec, 3))
print("  BALANCED accuracy   :", round(bal, 3))
print("  do-nothing baseline : accuracy 0.779, balanced 0.500")

over 5 honest splits, mean:
  plain accuracy      : 0.816
  recall (defaulters) : 0.326
  specificity (payers): 0.961
  BALANCED accuracy   : 0.644
  do-nothing baseline : accuracy 0.779, balanced 0.500
